# 01 · Prepare data: videos → face clips (once)

Downloads FF++ c23 videos **into the Colab runtime** (not Drive), cuts face clips and stores only these in Drive:
`DRIVE_ROOT/cache/<CACHE_VERSION>/` (one `.npz` per video + `index.csv` + `config.json`).

* Restartable: videos that are already in the cache are skipped. If Colab disconnects, run everything again.
* A CPU runtime is enough for this notebook.
* Different crop settings → new `CACHE_VERSION` (the code refuses to mix settings in one cache).

In [ ]:
#@title 0 · Setup (run after every restart)
DRIVE_ROOT = "/content/drive/MyDrive/IS665_Deepfake"  #@param {type:"string"}
REPO_URL = "https://github.com/payelsarkarr/is665-team6-deepfake-recognition"  #@param {type:"string"}
from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
os.makedirs(DRIVE_ROOT, exist_ok=True)
REPO = "/content/repo"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)       # newest code from GitHub
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO], check=True)
sys.path.insert(0, f"{REPO}/src")
import cv2
if not hasattr(cv2, "CascadeClassifier"):
    !pip -q install "opencv-python-headless<5"
    print("OpenCV 4 installed -> Runtime > Restart session, then run this cell again.")
META = "/content/FaceForensics/dataset"            # official split files
if not os.path.exists(META):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/ondyari/FaceForensics.git", "/content/FaceForensics"], check=True)
import importlib, ffpp
for m in ["data", "crops", "dataset", "models", "metrics", "train", "evaluate", "download"]:
    importlib.reload(importlib.import_module(f"ffpp.{m}"))
print("code:", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())

In [ ]:
#@title 1 · Settings
DATA_SOURCE = "official"  #@param ["official", "kaggle"]
#@markdown Official download script (from the FF++ email), stored in your Drive — never in the repo:
SCRIPT = f"{DRIVE_ROOT}/faceforensics_download_v4.py"  #@param {type:"string"}
#@markdown 0 = all 1,000 originals (5,000 videos). 250 = interim subset (1,250 videos).
MAX_PER_CLASS = 0  #@param {type:"integer"}
CACHE_VERSION = "crops_v1"  #@param {type:"string"}
CLIPS_PER_VIDEO = 4  #@param {type:"integer"}
FRAMES_PER_CLIP = 16  #@param {type:"integer"}
VIDEO_DIR = "/content/ffpp"
CACHE_DIR = f"{DRIVE_ROOT}/cache/{CACHE_VERSION}"

In [ ]:
#@title 2 · Download videos into the runtime (about 9–10 GB for all 5,000)
from ffpp import download
if DATA_SOURCE == "official":
    assert os.path.exists(SCRIPT), f"Put the official download script at {SCRIPT}"
    status = download.official(SCRIPT, VIDEO_DIR, num_videos=MAX_PER_CLASS)
    print(status)
else:
    !pip -q install kagglehub
    VIDEO_DIR = str(download.kaggle())
print("videos in", VIDEO_DIR)

`num_videos` of the official script takes the first N videos of its own file list, and the fakes it picks then match those originals. With the Kaggle mirror all files are downloaded and the subset is chosen in the next cell.

In [ ]:
#@title 3 · Manifest: labels, official splits, leakage exclusion
from ffpp import data
man = data.build_manifest(VIDEO_DIR, META, max_per_class=MAX_PER_CLASS)
print(len(man), "videos | not used:", man.attrs["skipped"])
print("excluded from val/test (leakage):", int(man.leak_excluded.sum()))
data.summary(man)

In [ ]:
#@title 4 · Extract face clips into Drive (restartable)
#@markdown About 1 s per video and CPU core; 5,000 videos ≈ 40–90 min on Colab.
from ffpp import crops
idx = crops.extract_all(man, CACHE_DIR, cfg=dict(clips_per_video=CLIPS_PER_VIDEO, frames_per_clip=FRAMES_PER_CLIP))
print(idx.status.value_counts().to_dict())
print("clips per video:", idx.n_clips.value_counts().sort_index().to_dict())
!du -sh "{CACHE_DIR}"

In [ ]:
#@title 5 · Look at a few clips (first, middle and last frame of clip 1)
import matplotlib.pyplot as plt, pandas as pd
ok = idx[idx.status == "ok"]
show = pd.concat([ok[ok.method == m].head(1) for m in ok.method.unique()]) if len(ok) else ok
fig, axes = plt.subplots(len(show), 3, figsize=(6, 2.1 * len(show)), squeeze=False)
for r, row in enumerate(show.itertuples()):
    a = crops.load_clips(f"{CACHE_DIR}/{row.cache_file}")
    for k, t in enumerate([0, a.shape[1] // 2, a.shape[1] - 1]):
        axes[r, k].imshow(a[0, t]); axes[r, k].axis("off")
    axes[r, 0].set_title(row.video_id, fontsize=8, loc="left")
plt.tight_layout(); plt.show()